# DPO stretch rung — preference optimization on top of the SFT adapter (gpt551)

Day-5 stretch rung 1. Takes the **canonical live-teacher SFT adapter (gpt551)** and runs **DPO** on
hybrid preference pairs, then measures whether preference optimization sharpens spec-adherence
**beyond SFT alone** — the question the plan poses for Day 5.

Pipeline (GPU runtime):
1. Clone `main`; load the **gpt551 SFT adapter** from Drive (`MyDrive/slm-deid-gpt551/`).
2. **Stage A (on-policy):** sample the SFT model over the train inputs; its genuine errors become
   the `rejected` side of real preference pairs.
3. **Stage B (backfill):** where the SFT model was already correct, synthesize `rejected` by
   perturbing the gold spans (over-tag / missed-name / wrong-boundary). Round-robined for coverage.
4. **DPO** on the hybrid pairs (frozen `configs/dpo.yaml`; ref = adapter-disabled base) → `outputs/dpo-<RUN>`.
5. Score **base vs SFT(gpt551) vs DPO** on the quarantined eval sets, print the deltas.
6. Persist adapter + pairs + reports to `MyDrive/slm-deid-dpo-<RUN>/`.

> **Hard ceilings honored.** Pairs come only from the eval-disjoint train split and no perturbation
> edits the passage text, so no eval surface can enter training — the notebook asserts
> `eval_leak_count == 0` before DPO. `configs/train.yaml` (the frozen SFT config) is **untouched**;
> `beta`/DPO-lr/epochs live in the separate `configs/dpo.yaml` and are new knobs, not SFT tuning.

> ⚠️ **TRL version.** The repo's `requirements.txt` pin (`trl>=0.9`) is stale — the DPO path needs a
> modern `DPOConfig`/`DPOTrainer` and Unsloth's `PatchDPOTrainer`. §0 installs a current `trl` and
> prints the version; if `PatchDPOTrainer` import or `DPOConfig(...)` fails, pin `trl` explicitly.

## 0. Setup — clone repo + install (GPU runtime)

Runtime → Change runtime type → **GPU** (T4 works; A100 is faster) before running.

In [ ]:
import os, sys

if not os.path.isdir("/content/slm-deid"):
    !git clone https://github.com/f15cubing/slm-deid.git /content/slm-deid
os.chdir("/content/slm-deid")

# This branch carries the DPO layer (src/train/prefs.py, src/train/dpo.py, configs/dpo.yaml).
!git fetch origin && git checkout -f worktree-dpo-stretch-rung && git reset --hard origin/worktree-dpo-stretch-rung
!echo "on commit:" && git log --oneline -1

sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

# QLoRA + DPO stack. trl is installed fresh (the requirements pin is stale — see the header note).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || pip install -q unsloth
!pip install -q "trl>=0.20" datasets faker pyyaml

In [ ]:
# Confirm a CUDA GPU + report the trl version the DPO path will actually run against.
!nvidia-smi -L
import torch, trl
assert torch.cuda.is_available(), "No CUDA GPU — set Runtime > Change runtime type > GPU."
print("CUDA device:", torch.cuda.get_device_name(0))
print("trl version:", trl.__version__)

# TRL's DPOTrainer import eagerly pulls optional integrations (mergekit, llm_blender, weave, ...).
# On Colab those often have leftover dist metadata so TRL's `is_*_available()` returns True, then the
# actual import fails or is pydantic-incompatible — crashing `from trl import DPOTrainer`. We use NONE
# of these, so disable every optional-integration availability flag BEFORE importing (keeping the core
# deps torch/transformers/peft/datasets/accelerate/bitsandbytes/rich). Ends the whack-a-mole.
import trl.import_utils as _tiu
_false = lambda *a, **k: False
_keep = ("torch", "transformers", "peft", "datasets", "accelerate", "bitsandbytes", "rich")
for _n in dir(_tiu):
    if _n.startswith("is_") and _n.endswith("_available") and not any(k in _n for k in _keep):
        setattr(_tiu, _n, _false)

from trl import DPOConfig, DPOTrainer            # must import cleanly
from unsloth import PatchDPOTrainer               # Unsloth's DPO memory patch
print("DPO imports OK (DPOConfig / DPOTrainer / PatchDPOTrainer)")

## 1. Mount Drive + locate the gpt551 SFT adapter

The gpt551 adapter + its exact train split live on Drive (git-ignored, per STATUS). Set
`SFT_RUN` to the run label whose adapter you want to start DPO from (default `gpt551`). The pairs are
built from **that run's own train split** so on-policy sampling reflects the data it was trained on.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SFT_RUN  = "gpt551"                                   # SFT run to DPO on top of
RUN_NAME = "gpt551"                                   # label for THIS DPO run's outputs

DRIVE_SFT = f"/content/drive/MyDrive/slm-deid-{SFT_RUN}"      # where the SFT run was persisted
SFT_LOCAL = f"outputs/sft-{SFT_RUN}"                          # local copy of the SFT adapter
DPO_OUT   = f"outputs/dpo-{RUN_NAME}"                         # this DPO run's adapter (VM-ephemeral)
DRIVE_OUT = f"/content/drive/MyDrive/slm-deid-dpo-{RUN_NAME}" # persistent per-run folder
os.makedirs(DRIVE_OUT, exist_ok=True)

# Copy the SFT adapter down locally (load_hf_tagger / train_dpo read a local dir).
import shutil
assert os.path.isdir(f"{DRIVE_SFT}/sft-v3"), f"SFT adapter not found at {DRIVE_SFT}/sft-v3 — check SFT_RUN/Drive."
shutil.copytree(f"{DRIVE_SFT}/sft-v3", SFT_LOCAL, dirs_exist_ok=True)

# Prefer the SFT run's OWN train split (the data it learned from); fall back to the committed one.
TRAIN_SPLIT = "data/splits/train.jsonl"
if os.path.exists(f"{DRIVE_SFT}/splits/train.jsonl"):
    os.makedirs("data/dpo_src", exist_ok=True)
    shutil.copy(f"{DRIVE_SFT}/splits/train.jsonl", "data/dpo_src/train.jsonl")
    TRAIN_SPLIT = "data/dpo_src/train.jsonl"
print("SFT adapter:", SFT_LOCAL, "| train split for pairs:", TRAIN_SPLIT)

## 2. Stage A — on-policy sampling of the SFT model

Run the SFT model over the train inputs; keep the rows where it **genuinely errs** as real
preference negatives. This is the highest-signal part — DPO learns against the model's actual
failure modes. Sequential generation, so ~`SAMPLE_N` forward passes (tens of minutes for the full
split on a T4; lower `SAMPLE_N` for a faster first pass).

In [ ]:
import time
from src.common.schema import read_jsonl
from src.infer import load_hf_tagger
from src.eval.behavioral_checks import check

rows = [ex.validate() for ex in read_jsonl(TRAIN_SPLIT)]
# Generation is SEQUENTIAL (one forward pass per input, no progress bar in tag_all), so this cell is
# the slow part: ~1-2s/example on an A100, several times that on a T4. Cap SAMPLE_N accordingly — a
# few hundred on-policy samples is plenty (Stage B backfills the rest). Raise it if you have an A100.
SAMPLE_N = min(len(rows), 400)
rows = rows[:SAMPLE_N]
print(f"sampling SFT over {len(rows)} train inputs (progress every 25; this is the slow cell)...")

sft = load_hf_tagger(adapter=SFT_LOCAL)          # 4-bit base + gpt551 LoRA
model_outputs = {}
t0 = time.time()
for i, ex in enumerate(rows, 1):
    model_outputs[ex.id] = sft.tag(ex.input)
    if i % 25 == 0 or i == len(rows):
        print(f"  {i}/{len(rows)}  ({(time.time() - t0) / i:.2f}s/ex)")

# How often did the SFT model actually err? (that's the on-policy pair supply)
n_err = sum(1 for ex in rows if not check(ex, model_outputs[ex.id]).passed)
print(f"SFT errors on train inputs: {n_err}/{len(rows)}  (these become on-policy negatives)")

del sft
import torch; torch.cuda.empty_cache()

## 3. Build the hybrid preference pairs

On-policy negatives where the model erred; deterministic backfill elsewhere. Asserts **zero eval leakage** before anything trains, then writes `data/dpo/pairs.jsonl`.

In [ ]:
import json
from collections import Counter
from src.train import prefs

PER_CAT_CAP = None            # set an int (e.g. 120) to balance categories; None = keep all
pairs = prefs.build_pairs(rows, model_outputs=model_outputs, per_category_cap=PER_CAT_CAP, seed=0)

leaks = prefs.eval_leak_count(pairs, "eval")
assert leaks == 0, f"EVAL LEAKAGE in preference pairs: {leaks} — investigate before training"

os.makedirs("data/dpo", exist_ok=True)
with open("data/dpo/pairs.jsonl", "w", encoding="utf-8") as f:
    for d in prefs.dump_pairs(pairs):
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

n_onpolicy = sum(1 for p in pairs if p.strategy == "on_policy")
print(f"pairs: {len(pairs)} | eval_leak: {leaks}")
print(f"on-policy (real SFT errors): {n_onpolicy} | deterministic backfill: {len(pairs) - n_onpolicy}")
print("by strategy:", dict(Counter(p.strategy for p in pairs)))
print("by category:", dict(Counter(p.category for p in pairs)))

## 4. DPO training (frozen `configs/dpo.yaml`) → `outputs/dpo-<RUN>`

Policy initialized from the SFT adapter; reference = adapter-disabled base (standard QLoRA-DPO). New knobs (`beta`/DPO-lr/epochs) live in `configs/dpo.yaml`; the SFT config is untouched.

In [ ]:
from src.train.dpo import load_config, train_dpo

cfg = load_config("configs/dpo.yaml")
cfg["sft_adapter"] = SFT_LOCAL              # start DPO from the gpt551 SFT adapter
cfg["pairs_path"]  = "data/dpo/pairs.jsonl"
dpo_adapter = train_dpo(cfg, smoke=False, output_dir=DPO_OUT)
print("DPO adapter:", dpo_adapter)

## 5. Eval — base vs SFT(gpt551) vs DPO

Each engine loaded **once**, scored across the quarantined sets on this same 4-bit runtime (so the
three columns are directly comparable). The Day-5 win condition is a DPO gain on gpt551's weakest
axes — **precision / over_tag / consistency** — without giving back recall.

In [ ]:
from src.eval.run import evaluate, compare, write_report, _load_examples
from src.infer import load_hf_tagger
import torch

SETS = ["eval/hardcases", "eval/adversarial", "eval/heldout_names", "eval/ood_probe"]
ENGINES = [("base", None), ("sft-gpt551", SFT_LOCAL), ("dpo", dpo_adapter)]

examples = {s: _load_examples(s) for s in SETS}
results = {s: {} for s in SETS}                       # split -> label -> report

for label, adapter in ENGINES:
    tagger = load_hf_tagger(adapter=adapter)
    for s in SETS:
        rep = evaluate(tagger, examples[s], label=label)
        write_report(rep, report_dir=f"outputs/dpo_reports/{os.path.basename(s)}")
        results[s][label] = rep
    del tagger; torch.cuda.empty_cache()

for s in SETS:
    reps = [results[s][lbl] for lbl, _ in ENGINES]
    print(f"\n===== {s}  ({len(examples[s])} examples) =====")
    print(compare(reps))

In [ ]:
# Per-category f5 / over_tag / consistency: SFT -> DPO on the hard cases (where the delta should show).
sft_r = results["eval/hardcases"]["sft-gpt551"]
dpo_r = results["eval/hardcases"]["dpo"]
print("category            f5 (sft->dpo)   over_tag (sft->dpo)")
for cat in sorted({e.category for e in examples["eval/hardcases"]}):
    a = sft_r.by_category.get(cat); b = dpo_r.by_category.get(cat)
    if a and b:
        print(f"  {cat:18s} {a.f5:.2f} -> {b.f5:.2f}     {a.over_tag_rate:.2f} -> {b.over_tag_rate:.2f}")
_c = lambda v: "n/a" if v is None else f"{v:.2f}"
print(f"\noverall consistency  sft {_c(sft_r.overall.consistency)} -> dpo {_c(dpo_r.overall.consistency)}")

## 6. Persist to Drive

Copy the DPO adapter, the preference pairs, and the eval reports to `MyDrive/slm-deid-dpo-<RUN>/` so nothing is lost when the VM recycles. Paste the `compare()` tables back to record them in `docs/results.md`.

In [ ]:
import shutil
for src_dir, name in [
    (DPO_OUT, "dpo_adapter"),
    ("data/dpo", "pairs"),
    ("outputs/dpo_reports", "eval_reports"),
]:
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, f"{DRIVE_OUT}/{name}", dirs_exist_ok=True)
        print("saved", src_dir, "->", f"{DRIVE_OUT}/{name}")
print(f"\nDone — DPO run '{RUN_NAME}' persisted to {DRIVE_OUT}.")